# 准备文档

In [5]:
from langchain.agents import create_agent
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 读取文档
with open("./resources/评估.md", "r", encoding='utf-8') as f:
    doc = f.read()

# 切分文档
headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,  #是否去除文本种的标题
)

documents = markdown_splitter.split_text(doc)

for i, document in enumerate(documents,start=1):
    document.id = f"doc_{i}"

# 准备知识库

In [6]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 初始化向量模型
ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:4b", dimensions=1024)

# 初始化VectorStore
vector_store = InMemoryVectorStore(embedding=ollama_embeddings, )

# 添加文档
vector_store.add_documents(documents)


['doc_1',
 'doc_2',
 'doc_3',
 'doc_4',
 'doc_5',
 'doc_6',
 'doc_7',
 'doc_8',
 'doc_9',
 'doc_10',
 'doc_11',
 'doc_12',
 'doc_13',
 'doc_14',
 'doc_15',
 'doc_16',
 'doc_17',
 'doc_18',
 'doc_19',
 'doc_20',
 'doc_21']

# 2-Step模式

In [13]:
from langchain.agents.middleware import dynamic_prompt,ModelRequest
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
# 定义动态提示词中间件
@dynamic_prompt
def rag_prompt(request:ModelRequest):
    # 1.拿到用户问题，也就是消息列表中的最后一条消息
    print("="*15,"user Message","="*15)
    user_input=request.state["messages"][-1].text
    print(user_input,"\n")

    # 2.检索知识片段，list[document]，拼接为字符串
    docs=vector_store.similarity_search(user_input,k=2)
    context="\n\n".join(doc.page_content for doc in docs)       #join() 是 Python 字符串的一个方法，专门用来把可迭代对象（比如这里的生成器）里的所有字符串，用指定的字符串连接起来。"\n\n" 是连接符，代表两个连续的换行。这意味着每两段文档内容之间，都会空出一整行。
    print("="*15,"Context","="*15)
    print(context,"\n")

    # 3.拼接新系统提示词
    return f"""
    你是一个教资考试专家，你根据上下文回答用户问题，不能自己编造答案，答案要简短，直接引用原文。
Context：{context}"""


In [14]:
# 初始化模型
model=init_chat_model(
    model="deepseek-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

# 定义agent
agent = create_agent(model=model, middleware=[rag_prompt], )


In [16]:
from langchain.messages import HumanMessage

# 测试
response = agent.invoke({"messages": [HumanMessage("论语中教育的目的是什么？")]})

for message in response["messages"]:
    message.pretty_print()

=============== user Message ===============
论语中教育的目的是什么？ 

=============== Context ===============
### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。

### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。 

================================ Human Message =================================

论语中教育的目的是什么？
================================== Ai Message ==================================

以完善人格为教育的首要目的，培养士和君子。


# Agentic RAG

In [1]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

# 1.初始化模型
model =init_chat_model(
    model="deepseek-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

# 2.定义工具
@tool
def retrieve_docs(query:str) -> str:
    """检索知识库，获取与用户提问相关的知识片段"""
    # 1.根据query检索知识片段
    documents = vector_store.similarity_search(query)

    # 2.拼接知识片段
    context = "\n\n".join([doc.page_content for doc in documents])

    # 3.返回拼接后的知识片段
    return context

# 3.定义agent，绑定model并注册tool
agent = create_agent(
    model=model,
    tools=[retrieve_docs],
    system_prompt="你是一个教资考试专家，回答与教资的相关问题时，必须调用工具检索文档，根据文档回答用户问题，不能自己编造答案，答案要简短，直接引用原文"
)

In [7]:
# 测试
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage("论语中教育的目的是什么？")]} )

for message in response['messages']:
    message.pretty_print()



================================ Human Message =================================

论语中教育的目的是什么？
================================== Ai Message ==================================
Tool Calls:
  retrieve_docs (call_00_2cPra6uiXxcCOIg3kspm8231)
 Call ID: call_00_2cPra6uiXxcCOIg3kspm8231
  Args:
    query: 论语 教育目的
================================= Tool Message =================================
Name: retrieve_docs

### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。

## 第二节 教育的定义  
### （一）教育的定义  
广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  
狭义的教育就是指学校教育。  
教育的要素：教育者、受教育者、教育影响（主要是教育内容）。

### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。

### （二）教育的属性  
教育的本质属性是有目的地培养人的社会活动。  
教育的永恒性。  
教育的历史性。  
教育的相对独立性。  
教育的长期性。  
教育的生产性。  
教育的民族

In [8]:
# 测试
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage("你好，介绍一下你自己")]} )

for message in response['messages']:
    message.pretty_print()



================================ Human Message =================================

你好，介绍一下你自己
================================== Ai Message ==================================

你好！我是教资考试助手，专门为你解答与教师资格考试相关的问题。我的回答会严格依据知识库中的文档内容，不会自行编造。

你可以问我关于教资考试的各类问题，比如报考条件、考试科目、备考要点、教育教学知识等。有需要的话，随时提问，我会先检索相关资料再为你解答。
